# Финальное обучение на train + val

Пользователь подтвердил разрешение train+val. Объединяем 4999+699=5698 размеченных изображений, не перемещая и не меняя raw. Это **финальный** этап после фиксации pipeline. Рецепт partial+stronger сохраняется; число эпох заранее фиксировано по исходному лучшему checkpoint (50:20, 101:21), исходный cosine T_max=25 сохранён.

Инициализация из ImageNet V2, не из обученного на этих данных best.pt: это refit выбранного рецепта на объединённой выборке. Не запускается автоматически. Нужен Kaggle GPU; это единственная тяжёлая часть плана. Никакой Accuracy на бывшей val, early stopping или best.pt; сохраняется **final.pt последней заданной эпохи**. Test только для опциональных predictions.

Два presets нужны для воспроизведения двух участников прежнего ансамбля; не обязательно обучать оба, если выбран одиночный pipeline. Новые reranker/ensemble варианты не следует автоматически добавлять в refit до анализа их результатов.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.final_refit import run_final_refit
from src.experiment import load_config

CONFIG = 'configs/resnet101_final_train_val.json'  # либо resnet50_final_train_val.json
RUN_REFIT = False  # включить только для финального запуска на Kaggle
PREDICT_TEST = False
configuration = load_config(CONFIG, PROJECT_ROOT)
display(configuration)


{'seed': 42,
 'notes': 'FINAL refit of selected partial+stronger recipe on train+val. Fixed 21 epochs from original best epoch; cosine T_max=25 retained; no held-out val metric or checkpoint selection. User confirmed train+val permitted.',
 'data': {'raw_dir': 'data/raw',
  'image_size': 256,
  'batch_size': 32,
  'num_workers': 2,
  'pin_memory': True,
  'normalization': {'mean': [0.485, 0.456, 0.406],
   'std': [0.229, 0.224, 0.225]},
  'augmentation': {'horizontal_flip': 0.5,
   'vertical_flip': 0.5,
   'color_jitter': {'p': 0.8,
    'brightness': 0.15,
    'contrast': 0.15,
    'saturation': 0.15,
    'hue': 0.02},
   'gamma': {'p': 0.25, 'range': [0.85, 1.15]},
   'corruption_probability': 0.65,
   'max_corruptions': 2,
   'gaussian_noise': {'p': 0.35, 'std': [0.005, 0.025]},
   'gaussian_blur': {'p': 0.25, 'sigma': [0.5, 2.5]},
   'jpeg_compression': {'p': 0.25, 'quality': [60, 95]},
   'resolution_degradation': {'p': 0.2, 'scale': [0.4, 1.0]}},
  'persistent_workers': True,
  'p

In [2]:
if RUN_REFIT:
    result = run_final_refit(configuration, project_root=PROJECT_ROOT,
                             predict_test=PREDICT_TEST, show_plots=True)
    display(result['summary'])
else:
    print('Final refit disabled. No training or test inference performed.')


Final refit disabled. No training or test inference performed.


Артефакты: artifacts/models/<id>/final.pt; logs/config.json, metadata.json, history.csv, summary.json, training.log, learning_curves.png. В experiments.csv добавляется строка с **пустой val_accuracy** и пометкой FINAL train+val. Если PREDICT_TEST=True, сохраняются test probabilities и submission.

Старые честные train/val результаты остаются в журнале отдельно. По ним уже выбран рецепт; новые refit веса нельзя оценивать на бывшей val как на независимой выборке.
